In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# training_code

In [ ]:
# ================================
# 0. GPU SETUP (KAGGLE)
# ================================
# Ensures GPU (T4 / P100) is being used

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU is enabled:", torch.cuda.get_device_name(0))
else:
    raise Exception("Enable GPU in Kaggle settings")

torch.backends.cudnn.benchmark = True


# ================================
# 1. IMPORT LIBRARIES
# ================================

import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
import cv2
import pandas as pd
import numpy as np
from ultralytics import YOLO


# ================================
# 2. DATASET CLASS
# ================================

class DrivingDataset(Dataset):
    def __init__(self, csv_file, transform=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = self.data.iloc[idx, 0]
        angle = self.data.iloc[idx, 1]

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(angle, dtype=torch.float32)


# ================================
# 3. TRANSFORMS
# ================================

transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])


# ================================
# 4. LOAD DATA
# ================================

dataset = DrivingDataset("/kaggle/input/your-dataset/driving_data.csv", transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)


# ================================
# 5. RESNET MODEL
# ================================

resnet = models.resnet18(weights="IMAGENET1K_V1")
resnet.fc = nn.Linear(resnet.fc.in_features, 1)
resnet = resnet.to(device)


# ================================
# 6. YOLO MODEL
# ================================

yolo_model = YOLO("yolov8n.pt")  # auto uses GPU


# ================================
# 7. YOLO FEATURE EXTRACTION
# ================================

def extract_yolo_features(results):
    boxes = results[0].boxes

    num_objects = len(boxes)
    max_area = 0.0

    for box in boxes:
        x1, y1, x2, y2 = box.xyxy[0]
        area = (x2 - x1) * (y2 - y1)
        max_area = max(max_area, area.item())

    return torch.tensor([num_objects, max_area], dtype=torch.float32)


# ================================
# 8. FUSION MODEL
# ================================

class FusionModel(nn.Module):
    def __init__(self):
        super(FusionModel, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.fc(x)


fusion_model = FusionModel().to(device)


# ================================
# 9. LOSS & OPTIMIZER
# ================================

criterion = nn.MSELoss()
optimizer = optim.Adam(
    list(resnet.parameters()) + list(fusion_model.parameters()), lr=1e-4
)


# ================================
# 10. TRAINING LOOP
# ================================

for epoch in range(5):
    resnet.train()
    fusion_model.train()

    for images, angles in loader:

        images = images.to(device, non_blocking=True)
        angles = angles.to(device).unsqueeze(1)

        batch_features = []

        for i in range(len(images)):
            img_np = images[i].permute(1, 2, 0).cpu().numpy()
            img_np = (img_np * 255).astype(np.uint8)

            # YOLO (runs on GPU internally)
            results = yolo_model(img_np, verbose=False)

            yolo_feat = extract_yolo_features(results)

            # ResNet
            resnet_out = resnet(images[i].unsqueeze(0)).detach().cpu().squeeze()

            combined = torch.cat((yolo_feat, resnet_out.unsqueeze(0)), dim=0)
            batch_features.append(combined)

        batch_features = torch.stack(batch_features).to(device)

        outputs = fusion_model(batch_features)

        loss = criterion(outputs, angles)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")


# ================================
# 11. INFERENCE
# ================================

def predict(image_path):
    image = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img_tensor = transform(img_rgb).unsqueeze(0).to(device)

    results = yolo_model(img_rgb, verbose=False)
    yolo_feat = extract_yolo_features(results)

    res_out = resnet(img_tensor).detach().cpu().squeeze()

    combined = torch.cat((yolo_feat, res_out.unsqueeze(0)), dim=0).unsqueeze(0).to(device)

    final_output = fusion_model(combined)

    return final_output.item()


# ================================
# 12. SAVE MODELS
# ================================

torch.save(resnet.state_dict(), "/kaggle/working/resnet.pth")
torch.save(fusion_model.state_dict(), "/kaggle/working/fusion.pth")


# ================================
# 13. SAMPLE TEST
# ================================

print("Prediction:", predict("/kaggle/input/your-dataset/test.jpg"))

In [1]:
import os
import shutil
from sklearn.model_selection import train_test_split

# Input paths
BASE_PATH = "/kaggle/input/datasets/barkataliarbab/udacity-self-driving-car-obstacles-dataset/export"
IMG_PATH = os.path.join(BASE_PATH, "images")
LBL_PATH = os.path.join(BASE_PATH, "labels")

# Output path
OUTPUT_PATH = "/kaggle/working/udacity_split"

# Create folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_PATH, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_PATH, split, "labels"), exist_ok=True)

# Get image list
images = [f for f in os.listdir(IMG_PATH) if f.endswith(".jpg")]

# Split 70/15/15
train_imgs, temp_imgs = train_test_split(images, test_size=0.30, random_state=42)
val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.50, random_state=42)

# Function to copy
def move_files(file_list, split):
    for file in file_list:
        img_src = os.path.join(IMG_PATH, file)
        lbl_src = os.path.join(LBL_PATH, file.replace(".jpg", ".txt"))

        img_dst = os.path.join(OUTPUT_PATH, split, "images", file)
        lbl_dst = os.path.join(OUTPUT_PATH, split, "labels", file.replace(".jpg", ".txt"))

        shutil.copy(img_src, img_dst)
        if os.path.exists(lbl_src):
            shutil.copy(lbl_src, lbl_dst)

# Execute
move_files(train_imgs, "train")
move_files(val_imgs, "val")
move_files(test_imgs, "test")

print("✅ Udacity YOLO dataset split done!")

✅ Udacity YOLO dataset split done!


In [3]:
import os
import shutil
from sklearn.model_selection import train_test_split

BASE_PATH = "/kaggle/input/datasets/kumaresanmanickavelu/lyft-udacity-challenge"
OUTPUT_PATH = "/kaggle/working/lyft_split"

# Create output structure
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_PATH, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_PATH, split, "masks"), exist_ok=True)

all_images = []
all_masks = []

# Loop through dataA → dataE
for folder in ["dataA", "dataB", "dataC", "dataD", "dataE"]:
    rgb_path = os.path.join(BASE_PATH, folder, folder, "CameraRGB")
    seg_path = os.path.join(BASE_PATH, folder, folder, "CameraSeg")

    if not os.path.exists(rgb_path):
        continue

    files = sorted(os.listdir(rgb_path))

    for f in files:
        all_images.append(os.path.join(rgb_path, f))
        all_masks.append(os.path.join(seg_path, f))

# Split 70/15/15
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    all_images, all_masks, test_size=0.30, random_state=42
)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs, temp_masks, test_size=0.50, random_state=42
)

# Copy function
def move_pairs(imgs, masks, split):
    for img, mask in zip(imgs, masks):
        name = os.path.basename(img)

        shutil.copy(img, os.path.join(OUTPUT_PATH, split, "images", name))
        shutil.copy(mask, os.path.join(OUTPUT_PATH, split, "masks", name))

# Execute
move_pairs(train_imgs, train_masks, "train")
move_pairs(val_imgs, val_masks, "val")
move_pairs(test_imgs, test_masks, "test")

print("✅ Lyft segmentation dataset split done!")

✅ Lyft segmentation dataset split done!
